<img src="../../../Individual-Contest/Chicken_Counting/figs/IOAI-Logo.png" alt="IOAI Logo" width="200" height="auto">

[IOAI 2025 (Pekin, Chine), concours individuel](https://ioai-official.org/china-2025)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SKonteye/IOAI-2025/blob/main/fr/Individual-Contest/Chicken_Counting/Chicken_Counting.ipynb)

# Comptage de poulets

## **1. Description du probleme**

En tant que responsable d'une equipe de recherche en IA collaborant avec des eleveurs de poulets Silkie, vous devez resoudre un probleme critique de l'elevage traditionnel en plein air. Un comptage precis du betail est essentiel a la fois pour les agriculteurs et pour les compagnies d'assurance, car des facteurs tels que les epidemies ou les attaques de predateurs peuvent fortement affecter le taux de survie de ces poulets en peu de temps. Meme si l'assurance aide a reduire les risques agricoles, le traitement des demandes d'indemnisation exige un comptage precis des pertes de betail. Vos eleveurs ont donc sollicite votre equipe afin de developper des systemes de comptage automatises plus precis. Le defi consiste a concevoir un modele optimise de comptage de poulets Silkie fonde sur des techniques d'estimation de densite, capable de fournir des nombres fiables pour soutenir a la fois la gestion des exploitations et les procedures d'assurance.

Votre equipe dispose d'un extracteur de caracteristiques preentraine pour des images de poulets Silkie, mais vous devez concevoir et entrainer le decodeur d'estimation de densite afin de produire une solution de comptage complete. Votre tache consiste a prolonger cette base en developpant une architecture de decodeur efficace et une strategie d'entrainement permettant d'obtenir des comptages fiables pour les agriculteurs et les assureurs.

La figure ci-dessous montre une image du jeu de donnees, ainsi que la distribution de densite reelle correspondante et une distribution de densite predite generee par le modele de base. La densite totale (somme des densites sur toutes les zones) est indiquee.

<img src="../../../Individual-Contest/Chicken_Counting/figs/Chicken Counting Fig 1.png" width="800">

## **2. Jeu de donnees**

La structure du jeu d'images de poulets Silkie fourni est la suivante :

```
datasets/
├── train/
│   └── Un jeu de donnees avec les champs :
│       ├── `image` : `PIL.Image` avec canaux RGB (3x720x1280)
│       └── `density` : un tableau 2D de forme 180x320
└── base.pth (modele preentraine)

os.environ.get("DATA_PATH")/
├── test_a/
│   └── Un jeu de donnees avec les champs :
│       └── `image` : `PIL.Image` avec canaux RGB (3x720x1280)
└── test_b/
    └── Un jeu de donnees avec les champs :
        └── `image` : `PIL.Image` avec canaux RGB (3x720x1280)
```

(1) Emplacement de l'ensemble d'entrainement : `datasets`. Les fichiers de ce dossier servent a l'ajustement fin du modele. Il contient un dossier `train`, qui stocke un jeu de donnees de 100 images avec leurs cartes de densite correspondantes.

(2) Ensemble de validation (`test_a`) et ensemble de test (`test_b`) : ils seront utilises pour evaluer les scores sur le Leaderboard A et le Leaderboard B, respectivement. Ils seront inaccessibles aux participants. Seul le score obtenu sur l'ensemble de test B sera utilise pour le classement final.

(3) Taille des jeux de donnees :

- Ensemble d'entrainement : 100 images.
- Ensemble de validation : 100 images.
- Ensemble de test : 100 images.

(4) L'ensemble de validation (`test_a`) et l'ensemble de test (`test_b`) ne sont pas visibles.

(5) En raison des limites de calcul, les cartes de densite d'entrainement sont remodelees en $1\times 180 \times 320$. **REMARQUE** : la carte de densite de sortie peut etre vue comme une matrice reelle 2D de forme $180\times 320$ ; la somme de toutes ses valeurs correspond au nombre de poulets.

## **3. Tache**

Votre tache consiste a entrainer votre propre modele a partir des donnees d'entrainement afin de predire des cartes de densite, et donc de permettre le comptage des poulets.

Vous pouvez etendre et optimiser le modele preentraine fourni afin d'ameliorer la precision des predictions de comptage. Le modele preentraine `base.pth` ne contient que les poids des quatre premieres couches du modele (le modele d'extraction de caracteristiques), et la fonction `load_pretrained_weights_partial` du code de reference permet de charger ces poids partiels dans votre modele. Vous pouvez construire un decodeur de densite `DensityDecoder` et le combiner avec le module d'extraction de caracteristiques preentraine pour former un modele complet de prediction des donnees.

```
class DensityDecoder(nn.Module):
    def __init__(self):
        #################################################
        # Votre code ici
        #################################################

    def forward(self, x):
        #################################################
        # Votre code ici
        #################################################
        return x
```

Vous pouvez egalement construire votre propre modele sans utiliser le modele preentraine fourni.

Cette tache est la continuation de Satellite Weather Forecasting. Petit rappel : un UNet remplit facilement la memoire GPU sans ingenierie de caracteristiques ; dans ce cas, un message d'erreur de memoire GPU sera signale.

Veuillez respecter les regles suivantes afin d'obtenir un score valide :

(1) Votre modele doit produire la carte de densite predite.

(2) En raison des limites de calcul, votre carte de densite de sortie doit etre remodelee en $180 \times 320$. C'est egalement la forme des cartes de densite cibles fournies dans le jeu d'entrainement.

## 4. Soumission

Veuillez soumettre un **`submission.ipynb`** contenant les elements suivants :

（1）**Code d'entrainement**

- Inclure l'ensemble du pipeline d'entrainement.

（2）**Code d'evaluation**
- Evaluer votre modele sur l'ensemble de validation et l'ensemble de test.

- La sortie doit etre enregistree sous la forme **`submission.npz`**. Il doit s'agir d'un fichier `npz` valide contenant deux tableaux `pred_a` et `pred_b`, chacun de forme `100x1x180x320` (le script d'evaluation acceptera aussi des predictions de forme `100x180x320` si vous choisissez de supprimer la dimension de canal).

  **Tout resultat ne respectant pas la taille specifiee sera considere comme invalide et recevra un score d'evaluation nul.**

- Chaque element de la carte de densite doit etre **superieur ou egal a zero** ; sinon, le score d'evaluation sera nul.

## **5. Notation**

Vous serez notes selon l'erreur relative moyenne de votre modele. L'erreur relative est definie par :

$$
\text{Relative Error} = \frac{|y_i - \hat{y}_i|}{|y_i|}
$$

ou $y_i$ est la densite totale reelle du $i$-eme echantillon et $\hat{y}_i$ la densite totale predite pour ce meme echantillon.

Votre score final avant normalisation sera calcule a partir de votre erreur relative moyenne de la maniere suivante :

$$
\text{Score} = \exp(-\frac{1}{n} \sum_{i=1}^{n} \frac{|y_i - \hat{y}_i|}{y_i})
$$

## **6. Baseline et ensemble d'entrainement**

- Vous trouverez ci-dessous la solution de reference de base.
- Le jeu de donnees se trouve dans le dossier `training_set`.
- Le meilleur score obtenu par le Comite scientifique pour cette tache est de 0.89 ; ce score est utilise pour l'unification des scores.
- Le score de base obtenu par le Comite scientifique pour cette tache est de 0.71 ; ce score est utilise pour l'unification des scores.

### Imports

In [ ]:
import random
import numpy as np
import torch

seed = 42

random.seed(seed)                  # generateur aleatoire integre de Python
np.random.seed(seed)               # NumPy
torch.manual_seed(seed)            # PyTorch (CPU)
torch.cuda.manual_seed(seed)       # PyTorch (un seul GPU)
torch.cuda.manual_seed_all(seed)   # PyTorch (tous les GPU)

# Garantit un comportement deterministe
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_from_disk
import logging
from torchvision import transforms
from tqdm import tqdm
import numpy as np
import math

# Les participants doivent monter les jeux de donnees "counting_problem_train(v1)" lors de la creation du noeud.
TRAIN_PATH = "./"  # adresse de l'ensemble d'entrainement et de base.pth
# L'ensemble d'entrainement est deploye automatiquement sur la machine de test.
# Votre notebook peut acceder a TRAIN_PATH meme si vous ne le montez pas avec le notebook.
TRAINING_SET = TRAIN_PATH + "train" # adresse de l'ensemble d'entrainement
BASE_MODEL_PATH = TRAIN_PATH + "base.pth"# adresse du fichier .pth
DTYPE = torch.float32
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
scale = 100.0

### Utilitaires de journalisation

In [ ]:
def logging_level(level='info'):
    str_format = '%(asctime)s - %(levelname)s: %(message)s'
    if level == 'debug':
        logging.basicConfig(level=logging.DEBUG, format=str_format, datefmt='%Y-%m-%d %H:%M:%S')
    elif level == 'info':
        logging.basicConfig(level=logging.INFO, format=str_format, datefmt='%Y-%m-%d %H:%M:%S')
    return logging


class BatchLossLogger:
    def __init__(self, log_interval=100):
        self.losses = []
        self.batch_number = 0
        self.log_interval = log_interval

    def log(self, loss):
        self.losses.append(loss)
        self.batch_number += 1
        if self.batch_number % self.log_interval == 0:
            logging.info(f'Batch No. {self.batch_number:7d} - loss: {loss:.6f}')

### Entrainer votre modele
#### Definition du modele
Les poids preentraines du modele `FeatureExtraction` sont fournis dans `base.pth` dans l'ensemble d'entrainement, avec une fonction permettant de les charger.
`ChickenCounting` est le modele complet.

In [ ]:
class FeatureExtraction(nn.Module):
    def __init__(self, in_channels=3):
        super(FeatureExtraction, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=3, padding=2, dilation=2)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=3, padding=2, dilation=2)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=2, dilation=2)
        self.conv4 = nn.Conv2d(128, 128, kernel_size=3, padding=2, dilation=2)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)


    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = self.pool4(x)

        return x

    def load_pretrained_weights_partial(self, weights_path, num_layers=4):
        save_model = torch.load(weights_path)
        partial_state_dict = {}
        expected_layers = [
            'feature_extraction.conv1.weight', 'feature_extraction.conv1.bias',
            'feature_extraction.conv2.weight', 'feature_extraction.conv2.bias',
            'feature_extraction.conv3.weight', 'feature_extraction.conv3.bias',
            'feature_extraction.conv4.weight', 'feature_extraction.conv4.bias',
        ]
        state_dict = {k.split('.', 1)[-1]: v for k, v in save_model.items() if k in expected_layers[:2 * num_layers]}
        model_dict = self.state_dict()

        for k in state_dict:
            if k in model_dict:
                partial_state_dict[k] = state_dict[k]
                print(k)

        self.load_state_dict(partial_state_dict)


class DensityDecoder(nn.Module): # Definissez votre decodeur ici.
    def __init__(self):
        super(DensityDecoder, self).__init__()
        self.conv5 = nn.Conv2d(in_channels=128, out_channels=1, kernel_size=3, padding=2, dilation=2)

    def forward(self, x):
        x = F.relu(self.conv5(x))
        return x


class ChickenCounting(nn.Module):
    def __init__(self):
        super(ChickenCounting, self).__init__()
        self.feature_extraction = FeatureExtraction()
        self.feature_decoder = DensityDecoder()

    def forward(self, x):
        x = self.feature_extraction(x)
        x = self.feature_decoder(x)
        return x

### Lecture du jeu de donnees
Lire le jeu d'entrainement

In [ ]:
from datasets import load_dataset

# Charger le jeu de donnees depuis Hugging Face
train_dataset = load_dataset("ioaihsc/Task2_Chicken_Counting_Train2", 
                            data_dir="train",
                            split="train")  

image_transform = transforms.Compose([
    transforms.ToTensor(),
])

def collate_fn(batch, scale=scale):
    return {
        "image": torch.stack([image_transform(item["image"]) for item in batch]),
        "density": torch.stack([torch.tensor(item["density"], dtype=DTYPE).unsqueeze(0) * scale for item in batch])
    }

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(train_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

### Lancer l'entrainement

In [ ]:
# Definition du processus d'entrainement
def train_chickenfcn(model, train_loader, val_loader, optimizer, scheduler, num_epochs, device, save_path):
    model.train()
    criterion_mse = torch.nn.MSELoss(reduction='sum').to(device)
    criterion_mae = torch.nn.L1Loss(reduction='sum').to(device)
    best_loss = float('inf')
    print(train_loader.__len__())

    for epoch in range(num_epochs):
        train_loss_mse = 0.0
        train_loss_mae = 0.0

        train_loader_tqdm = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{num_epochs}', leave=False)

        for i, data in enumerate(train_loader_tqdm, 0):
            inputs, targets = data["image"], data["density"]
            inputs = inputs.to(device).float()
            targets = targets.to(device).float()
            # print(targets.shape)
            # t = np.sum((targets[0] / scale).cpu().numpy().squeeze())
            # print(t)

            optimizer.zero_grad()
            
            outputs = model(inputs)
            
            loss_mse = criterion_mse(outputs, targets)
            loss_mae = criterion_mae(outputs, targets)
            loss_mae.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)

            optimizer.step()
            train_loss_mse += loss_mse.item()
            train_loss_mae += loss_mae.item()

            train_loader_tqdm.set_postfix({'Train MSE Loss': loss_mse.item(), 'Train MAE Loss': loss_mae.item()})

        train_loss_mse /= (len(train_loader))
        train_loss_mae /= (len(train_loader))
        logging.info(
            f'Epoch [{epoch + 1}/{num_epochs}], Train MSE loss: {train_loss_mse:.8f}, MAE loss: {train_loss_mae:.8f}')

        scheduler.step()

        # Validation
        model.eval()
        val_loss_mse = 0.0
        val_loss_mae = 0.0

        val_loader_tqdm = tqdm(val_loader, desc=f'Validation Epoch {epoch + 1}/{num_epochs}', leave=False)

        with torch.no_grad():
            for i, data in enumerate(val_loader_tqdm, 0):
                inputs, targets = data["image"], data["density"]
                inputs = inputs.to(device).float()
                targets = targets.to(device).float()

                outputs = model(inputs)
                mse_loss = criterion_mse(outputs, targets)
                mae_loss = criterion_mae(outputs, targets)
                val_loss_mse += mse_loss.item()
                val_loss_mae += mae_loss.item()

                val_loader_tqdm.set_postfix(
                    {'Validation MSE Loss': mse_loss.item(), 'Validation MAE Loss': mae_loss.item()})

        val_loss_mse /= (len(val_loader))
        val_loss_mae /= (len(val_loader))
        logging.info(
            f'Epoch [{epoch + 1}/{num_epochs}], Validation MSE Loss: {val_loss_mse:.8f}, MAE Loss: {val_loss_mae:.8f}')

        # Enregistrer le modele
        if val_loss_mae < best_loss:
            best_loss = val_loss_mae
            torch.save(model.state_dict(), save_path)

    print('Finished Training ChickenFCN')

In [ ]:
logging = logging_level('info')
logging.debug('use debug level logging setting')

################################################################################
# Parametres de l'experience
################################################################################
learning_rate = 1e-4
lr_decay = 1e-5
weight_decay = 0.0001
save_path = "model.pth"

epochs = 20

# Entrainement
model = ChickenCounting().to(DEVICE)
model.feature_extraction.load_pretrained_weights_partial(BASE_MODEL_PATH)
print('load model success')

optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=1 - lr_decay)

logging.info('Begin training single view model...')
train_chickenfcn(model, train_loader, val_loader, optimizer, scheduler, epochs, DEVICE, save_path=save_path)
logging.info('Finished training single view model.')

### Evaluer le modele
Cette section valide le modele sur l'ensemble d'entrainement, ce qui aide les participants a comprendre si le modele est utilisable et a calculer le score sur l'ensemble d'entrainement.

In [ ]:
# Definition de la fonction d'evaluation
def evaluate(model, val_loader, device, scale): # Fonction utilisee pour la notation finale.
    model.eval()  # Placer le modele en mode evaluation

    # Initialiser les metriques
    mse = 0.0
    mae = 0.0
    predict_num = 0.0
    true_num = 0.0
    rate = 0.0

    with torch.no_grad():  # Desactiver le calcul des gradients pour l'inference
        for i, data in enumerate(val_loader, 0):
            inputs, targets = data["image"], data["density"]
            inputs = inputs.to(device).float()  # Deplacer les entrees vers le device et convertir en float
            targets = targets.to(device).float()  # Deplacer les cibles vers le device et convertir en float

            # Obtenir les predictions du modele
            outputs = model(inputs) / scale  # Ajuster selon le facteur d'echelle

            # Convertir les tenseurs en numpy pour la visualisation et le calcul des metriques
            inputs_np = inputs.cpu().numpy()  # Convertir les entrees en numpy
            targets_np = targets.cpu().numpy()  # Convertir les cibles en numpy
            outputs_np = outputs.cpu().numpy()  # Convertir les sorties en numpy
            # imshow_res(inputs_np, targets_np, outputs_np, scale)  # Decommenter pour visualiser les resultats

            # Calculer les sommes reelles et predites pour comparaison
            t = np.sum((targets[0] / scale).cpu().numpy().squeeze())  # Somme verite terrain
            g = np.sum(outputs.cpu().numpy().squeeze())  # Somme predite
            print(f'NO.{i}   true_sum={t}, get_sum={g}, abs={abs(t - g)}, rate={abs(1 - g / t)}')

            # Mettre a jour les metriques
            predict_num += g
            true_num += t
            rate += abs(1 - g / t)
            mae += abs(t - g)
            mse += abs(t - g) * abs(t - g)

    # Calculer les metriques moyennes sur tous les batches
    mae /= len(val_loader)
    mse /= len(val_loader)
    predict_num /= len(val_loader)
    true_num /= len(val_loader)
    rate /= len(val_loader)

    # Journaliser les resultats
    logging.info(
        f'test ---- Score: {math.exp(-rate):.3f}, MSE: {mse:.4f}, MAE: {mae:.4f}, Chicken_avg: {predict_num:.4f}')
    return math.exp(-rate)

In [ ]:
model.load_state_dict(torch.load(save_path, map_location=DEVICE))
model.to(DEVICE)
evaluate(model, val_loader, DEVICE, scale)

### Soumission
Cette partie genere les fichiers de resultat destines aux tests et a la notation.
Les participants ne pouvaient pas acceder localement a l'ensemble de validation (`test_a`) ni a l'ensemble de test (`test_b`).
Lisez attentivement le code suivant et veillez a respecter les conventions de nommage des fichiers.

In [ ]:
from datasets import load_dataset

test_dataset = load_dataset("ioaihsc/Task2_Chicken_Counting_Test", 
                            data_dir="valandtest",
                            split="validation")

def collate_fn(batch): # Les jeux de test ne fourniront pas les densites cibles
    return torch.stack([image_transform(item["image"]) for item in batch])

test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

predictions = []
model.eval()
with torch.no_grad():
    for batch in tqdm(test_loader):
        outputs = model(batch.to(DEVICE)) / scale
        predictions.append(outputs.cpu().numpy())

pred_a = np.concatenate(predictions, axis=0)

del test_dataset
del test_loader
del predictions

test_dataset = load_dataset("ioaihsc/Task2_Chicken_Counting_Test", 
                            data_dir="valandtest",
                            split="test")  
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

predictions = []
with torch.no_grad():
    for batch in tqdm(test_loader):
        outputs = model(batch.to(DEVICE)) / scale
        predictions.append(outputs.cpu().numpy())

pred_b = np.concatenate(predictions, axis=0)

np.savez('submission.npz', pred_a=pred_a, pred_b=pred_b) # enregistrer vos soumissions dans `submission.npz` avec les cles `pred_a` et `pred_b`

In [ ]:
%run ./metrics.py